# Batch Inference with Champion Model

This notebook deploys the champion model from the Azure ML model registry and performs batch inference on data loaded from blob storage.

In [24]:
import json
import pandas as pd
import joblib
from pathlib import Path
from azureml.core import Workspace, Model, Dataset, Datastore
from azureml.data import OutputFileDatasetConfig
from azureml.data.datapath import DataPath
import os
import shutil
from sklearn.metrics import classification_report

In [25]:
# Configuration
MODEL_NAME = "CreditCard_Fraud_Detection"
BATCH_INPUT_FILE = "creditcard_batch_input.csv"
BATCH_DATASTORE_PATH = 'UI/2026-04-17_180302_UTC/' + BATCH_INPUT_FILE
BASE_DIR = Path('/home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure')
OUTPUTS_DIR = BASE_DIR / 'outputs'
BATCH_RESULTS_DIR = OUTPUTS_DIR / 'batch_results'
BATCH_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
# Connect to Azure ML Workspace
ws = Workspace.from_config()
print(f"Connected to workspace: {ws.name}")

Connected to workspace: azuremlops


In [27]:
# Get the champion model from registry
def get_champion_model(ws, model_name):
    models = Model.list(ws, name=model_name)
    champion_models = [m for m in models if m.tags.get('role') == 'champion']
    if not champion_models:
        raise ValueError("No champion model found in registry")
    # Get the latest champion
    champion = max(champion_models, key=lambda m: int(m.version))
    print(f"Found champion model: {champion.name} v{champion.version}")
    return champion

champion_model = get_champion_model(ws, MODEL_NAME)

Found champion model: CreditCard_Fraud_Detection v4


In [28]:
# Download the champion model locally
model_local_path = OUTPUTS_DIR / 'champion_model'
model_local_path.mkdir(exist_ok=True)
champion_model.download(target_dir=str(model_local_path), exist_ok=True)
print(f"Champion model downloaded to: {model_local_path}")

# Load the model
model_file = model_local_path / 'model.pkl'
if not model_file.exists():
    # Sometimes it's directly the model file
    model_files = list(model_local_path.glob('*.pkl'))
    if model_files:
        model_file = model_files[0]
    else:
        raise FileNotFoundError(f"Model file not found in {model_local_path}")

model = joblib.load(model_file)
print("Champion model loaded successfully")

Champion model downloaded to: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/champion_model
Champion model loaded successfully


In [29]:
# Load batch input data from blob storage using the same Azure ML method as Model_Training.ipynb
# Reference the workspace blob datastore and the batch input file path

datastore = Datastore.get(ws, 'workspaceblobstore')
print(f"Using datastore: {datastore.name}")

batch_dataset = Dataset.File.from_files(path=DataPath(datastore, BATCH_DATASTORE_PATH))
downloaded_paths = batch_dataset.download(target_path=str(BASE_DIR / 'data'), overwrite=True)

data_path = downloaded_paths[0]
df = pd.read_csv(data_path)
print(f"Loaded batch input from: {data_path}")
print(f"Original data shape: {df.shape}")

Using datastore: workspaceblobstore
{'infer_column_types': 'False', 'activity': 'download'}
{'infer_column_types': 'False', 'activity': 'download', 'activityApp': 'FileDataset'}
Loaded batch input from: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/data/creditcard_batch_input.csv
Original data shape: (134807, 31)


In [30]:
# Use all observations for batch inference
df_sample = df.copy()
print(f"Using all {len(df_sample)} observations for batch inference")
print(f"Data shape: {df_sample.shape}")
print(df_sample.head())


Using all 134807 observations for batch inference
Data shape: (134807, 31)
       ID     Time        V1        V2        V3        V4        V5  \
0  150001  92349.0 -3.833346  3.178578 -2.271357 -0.965729 -0.426073   
1  150002  92349.0  2.007115  0.113504 -1.495139  0.567020  0.340002   
2  150003  92351.0 -0.669129  1.336780 -0.285828 -0.035038  0.296780   
3  150004  92352.0 -0.818886  1.066989  1.114803 -0.355338 -0.258985   
4  150005  92353.0  1.808302 -0.499924 -2.100796 -0.112869  2.046647   

         V6        V7        V8  ...       V20       V21       V22       V23  \
0 -0.603747  0.227831  0.838212  ...  0.456224 -0.626699 -0.656796 -0.059550   
1 -0.643120 -0.079727 -0.114251  ... -0.258032 -0.460822 -1.088131  0.370633   
2 -1.076220  0.469643  0.323898  ... -0.237969 -0.095581  0.095008 -0.176772   
3  1.455683 -1.211008 -2.902691  ... -0.801534  2.613335 -2.123401 -0.101610   
4  3.703385 -1.054428  0.928300  ...  0.033840 -0.374936 -1.006065  0.368108   

        V24

In [31]:
# Prepare data for inference (assuming same preprocessing as training)
# Drop target column if present
target_col = 'Class'  # Assuming the target column name
if target_col in df_sample.columns:
    X = df_sample.drop(columns=[target_col])
    y_true = df_sample[target_col]
    has_target = True
    print("Target column found, will compute metrics")
else:
    X = df_sample
    y_true = None
    has_target = False
    print("No target column found, predictions only")

print(f"Features shape: {X.shape}")

No target column found, predictions only
Features shape: (134807, 31)


In [32]:
# Perform batch inference
print("Performing batch inference...")
predictions = model.predict(X)
probabilities = model.predict_proba(X)[:, 1]  # Probability of fraud (class 1)

print(f"Predictions shape: {predictions.shape}")
print(f"Unique predictions: {pd.Series(predictions).value_counts()}")

Performing batch inference...
Predictions shape: (134807,)
Unique predictions: 0    134639
1       168
dtype: int64


In [33]:
# Create results dataframe
results_df = df_sample.copy()
results_df['prediction'] = predictions
results_df['fraud_probability'] = probabilities

# Save batch results
results_file = BATCH_RESULTS_DIR / 'batch_predictions.csv'
results_df.to_csv(results_file, index=False)
print(f"Batch predictions saved to: {results_file}")

# Save summary
summary = {
    'total_observations': len(results_df),
    'fraud_predictions': int(sum(predictions)),
    'fraud_percentage': float(sum(predictions) / len(predictions) * 100),
    'model_version': champion_model.version,
    'model_name': champion_model.name
}

if has_target:
    report = classification_report(y_true, predictions, output_dict=True)
    summary['classification_report'] = report
    print("\nClassification Report:")
    print(classification_report(y_true, predictions))

summary_file = BATCH_RESULTS_DIR / 'batch_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Batch summary saved to: {summary_file}")

print("\nBatch inference completed successfully!")
print(f"Results: {summary}")

Batch predictions saved to: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/batch_results/batch_predictions.csv
Batch summary saved to: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/batch_results/batch_summary.json

Batch inference completed successfully!
Results: {'total_observations': 134807, 'fraud_predictions': 168, 'fraud_percentage': 0.12462260861824682, 'model_version': 4, 'model_name': 'CreditCard_Fraud_Detection'}
